# Data Extraction
The sales files are arranged strangely as usual, with multiple files per year, and sometime with checkpoints.  
I'm going to follow the same strategy as Gwinnett, where I just take all of the sales data and deduplicate at the end.  

**Selected Data Files**  
These were all taken from the Dekalb-Hicks folder of the County Tax Assessment Data Folder  
Dekalb Count Full File2015/SALES.CSV  
Dekalb Count Full File EOY2015/SALES.CSV  
Dekalb Count Full File EOY2016/SALES.CSV  
Dekalb Count Full File EOY2017/SALES.CSV  
Dekalb Count Full File Oct 2018/SALES.CSV  
Dekalb Count Full File EOY2019/SALES.CSV  
Dekalb Count Full File EOY2020/SALES.CSV  
Dekalb Count Full File Aug 2021/SALES.CSV  
2013 Full File CD 2003 as of 8-1-2013.mdb  
2014 Full File CD 2003 as of 8-13-2014.mdb

They were all renamed to the format SALES_20{XX}.CSV for convenience

In [11]:
import pandas as pd
import os

DATA_PATH = "../../data/dekalb"
OUT_PATH = "../../data/dekalb/out"

In [103]:
year_dfs = []

for file_p in os.listdir(DATA_PATH):
    if file_p.endswith(".csv") and file_p.startswith("SALES"):
        year = int(file_p.split("_")[1].split(".")[0])
        df = pd.read_csv(os.path.join(DATA_PATH, file_p))

        if year > 2014:
            df = df.rename(columns={x: x.strip() for x in df.columns})
        else:
            df = df.rename(columns={'Parid': 'PARID', 'Book': "BOOK", 'Page': "PAGE", 'Oldown': "OLDOWN", 'Own1': "OWN1",
                            'Saledt': "SALEDT", 'Price': "PRICE", 'Saletype': "SALETYPE", 'Instrtyp': "INSTRTYP",
                            'Saleval': "SALEVALDESCR"})[['PARID', 'BOOK', 'PAGE', 'OLDOWN', 'OWN1', 'SALEDT',
                                                         'PRICE', 'SALETYPE', 'INSTRTYP', 'SALEVALDESCR']]

        year_dfs.append((year, df))

/var/folders/bb/g7vlcgfn14ld8_w41331g0dw0000gn/T/ipykernel_15328/853511927.py:6: DtypeWarning: Columns (9,16,27,28,42) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(DATA_PATH, file_p))
/var/folders/bb/g7vlcgfn14ld8_w41331g0dw0000gn/T/ipykernel_15328/853511927.py:6: DtypeWarning: Columns (16,21,24,25,26,27,28,30,33,41,42,43) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(DATA_PATH, file_p))


In [104]:
year_dfs.sort(key = lambda x : x[0])

In [106]:
for year, df in year_dfs:
    print(year)
    print(df.columns)

2013
Index(['PARID', 'BOOK', 'PAGE', 'OLDOWN', 'OWN1', 'SALEDT', 'PRICE',
       'SALETYPE', 'INSTRTYP', 'SALEVALDESCR'],
      dtype='object')
2014
Index(['PARID', 'BOOK', 'PAGE', 'OLDOWN', 'OWN1', 'SALEDT', 'PRICE',
       'SALETYPE', 'INSTRTYP', 'SALEVALDESCR'],
      dtype='object')
2015
Index(['PARID', 'BOOK', 'PAGE', 'OLDOWN', 'OWN1', 'SALEDT', 'PRICE',
       'SALETYPE', 'INSTRTYP', 'SALEVALDESCR'],
      dtype='object')
2016
Index(['PARID', 'BOOK', 'PAGE', 'OLDOWN', 'OWN1', 'SALEDT', 'PRICE',
       'SALETYPE', 'INSTRTYP', 'SALEVALDESCR'],
      dtype='object')
2017
Index(['PARID', 'BOOK', 'PAGE', 'OLDOWN', 'OWN1', 'SALEDT', 'PRICE',
       'SALETYPE', 'INSTRTYP', 'SALEVALDESCR'],
      dtype='object')
2018
Index(['PARID', 'BOOK', 'PAGE', 'OLDOWN', 'OWN1', 'SALEDT', 'PRICE',
       'SALETYPE', 'INSTRTYP', 'SALEVALDESCR'],
      dtype='object')
2019
Index(['PARID', 'BOOK', 'PAGE', 'OLDOWN', 'OWN1', 'SALEDT', 'PRICE',
       'SALETYPE', 'INSTRTYP', 'SALEVALDESCR'],
      dtype='o

Luckily looks like the entire thing has the same data format.

In [107]:
year_df_list = []
for year, year_df in year_dfs:
    for column in year_df.columns:
        if year_df.dtypes[column] == object:
            year_df[column] = year_df[column].str.strip()

    year_df['SALEDT'] = pd.to_datetime(year_df['SALEDT'], format="%d-%b-%y" , errors="coerce")
    year_df['SALE_YR'] = year_df["SALEDT"].dt.year
    year_df_list.append(year_df)

In [109]:
for year, year_df in year_dfs:
    year_df.loc[year_df['SALE_YR'] > year, 'SALE_YR'] -= 100
    year_df.loc[year_df['SALE_YR'] > year, 'SALEDT'] -= pd.offsets.DateOffset(years=100)

In [110]:
for year, year_df in year_dfs:
    print(year)
    print(year_df['SALE_YR'].value_counts().sort_index())

2013
SALE_YR
1914        3
1915        1
1916        2
1917        8
1918        4
        ...  
2009    31614
2010    33993
2011    30358
2012    32742
2013    10760
Name: count, Length: 99, dtype: int64
2014
SALE_YR
1953        1
1960        1
1961        1
1963        1
1964        1
1965        7
1966        2
1967        1
1969        3
1970        7
1971        3
1972        7
1973        9
1974        8
1975        6
1976        7
1977       12
1978        6
1979       11
1980       15
1981        8
1982       27
1983       14
1984       17
1985       37
1986       49
1987       15
1988       20
1989       17
1990       17
1991       25
1992       15
1993       18
1994       21
1995       41
1996       26
1997       45
1998       42
1999       46
2000       80
2001       61
2002       80
2003       65
2004      101
2005      119
2006      167
2007      195
2008      177
2009      169
2010      577
2011     5279
2012    32354
2013    28607
2014    11207
Name: count, dtype: int64


In [111]:
sales_digest_df = pd.concat(year_df_list)

In [112]:
old_rows = sales_digest_df.shape[0]
sales_digest_df_dedup = sales_digest_df.drop_duplicates(subset=['PARID', 'BOOK', 'PAGE', 'SALEDT'])

new_rows = sales_digest_df_dedup.shape[0]

print(f"{old_rows - new_rows} rows removed. {new_rows} Remaining.")

24460 rows removed. 1297214 Remaining.


In [113]:
sales_digest_df_dedup = sales_digest_df_dedup.sort_values(by = "SALEDT")

In [114]:
sales_digest_df_dedup.to_csv(os.path.join(OUT_PATH, "DEKALB_SALES_FINAL.csv"), index=False)

In [82]:
tax_digest_df = pd.read_csv(os.path.join(DATA_PATH, "dekalb_digest_withnonprofit.csv"), low_memory=False)

In [83]:
tax_digest_df.describe()

,TAXYR,ASMT_TAXYR,ASMT_FMV_LAND,ASMT_FMV_BLDG,LAND_TAXYR,LAND_LLINE,LAND_SF,LAND_ACRES,LAND_UNITS,LAND_BRATE,...,DWELL_WBFP_O,DWELL_WBFP_S,DWELL_WBFP_PF,DWELL_BSMTCAR,DWELL_SFLA,is_non_owner_occupied,ADRNO_property_x,ADRNO_property_y,owner_mail_paddr2,rental_flag
count,2.384941e+06,2.148622e+06,2.148622e+06,2.148622e+06,2.146382e+06,2.146382e+06,2.109683e+06,2.109588e+06,2.105139e+06,2.146382e+06,...,1.450344e+06,1.450562e+06,17070.000000,5672.000000,1.884574e+06,1.914245e+06,1.879489e+06,2.349390e+06,0.0,2.384941e+06
mean,2.017532e+03,2.017811e+03,7.679382e+04,1.900591e+05,2.017810e+03,1.000311e+00,1.410724e+04,3.255247e-01,1.381800e+00,3.638300e+04,...,7.182710e-01,7.181768e-01,1.032513,1.827574,1.927447e+03,9.999979e-01,2.801931e+03,2.801923e+03,NaN,4.647452e-01
std,2.874494e+00,2.896107e+00,7.543695e+05,1.434179e+06,2.896314e+00,1.818495e-02,4.522374e+05,1.470531e+01,1.580440e+02,2.996436e+04,...,5.554250e-01,5.449679e-01,0.533321,0.402200,7.810139e+02,1.445543e-03,1.792883e+03,1.793132e+03,NaN,4.987557e-01
min,2.013000e+03,2.013000e+03,0.000000e+00,0.000000e+00,2.013000e+03,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN,0.000000e+00
25%,2.015000e+03,2.016000e+03,5.600000e+03,0.000000e+00,2.016000e+03,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.110000e+04,...,0.000000e+00,0.000000e+00,1.000000,2.000000,1.362000e+03,1.000000e+00,1.395000e+03,1.395000e+03,NaN,0.000000e+00
50%,2.018000e+03,2.018000e+03,2.400000e+04,1.036000e+05,2.018000e+03,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,3.210000e+04,...,1.000000e+00,1.000000e+00,1.000000,2.000000,1.758000e+03,1.000000e+00,2.568000e+03,2.568000e+03,NaN,0.000000e+00
75%,2.020000e+03,2.020000e+03,7.160000e+04,1.918000e+05,2.020000e+03,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,5.000000e+04,...,1.000000e+00,1.000000e+00,1.000000,2.000000,2.333000e+03,1.000000e+00,3.960000e+03,3.960000e+03,NaN,1.000000e+00
max,2.022000e+03,2.022000e+03,2.812390e+08,3.144764e+08,2.022000e+03,3.000000e+00,1.457953e+08,1.475300e+04,8.712000e+04,4.596774e+06,...,2.100000e+01,9.000000e+00,8.000000,3.000000,2.000000e+04,1.000000e+00,1.930400e+04,1.930400e+04,NaN,1.000000e+00


In [115]:
sales_digest_eligible_df = sales_digest_df_dedup[(sales_digest_df_dedup['SALE_YR'] <= 2022) & (sales_digest_df_dedup['SALE_YR'] >= 2013)]
sales_tax_merged = pd.merge(sales_digest_eligible_df, tax_digest_df, how="left", left_on=["PARID", "SALE_YR"], right_on=["PARID", "TAXYR"])

In [116]:
pct_merged = sales_tax_merged[sales_tax_merged["TAXYR"].notna()].shape[0] / sales_tax_merged.shape[0]
print(f"Pct Succesfully Merged: {pct_merged}")

Pct Succesfully Merged: 0.9925136014627566


In [118]:
sales_tax_merged.to_csv(os.path.join(OUT_PATH, "DEKALB_SALES_TAX_DIGEST_FINAL.csv"), index=False)

In [120]:
unmerged_pids = pd.Series(sales_tax_merged[sales_tax_merged["TAXYR"].isna()]["PARID"].unique())

In [124]:
unmerged_not_present = unmerged_pids[~unmerged_pids.isin(tax_digest_df["PARID"])]
unmerged_present = unmerged_pids[unmerged_pids.isin(tax_digest_df["PARID"])]
print(unmerged_not_present.shape)
print(unmerged_present.shape)

(9,)
(1022,)


In [123]:
unmerged_not_present_df = sales_tax_merged[sales_tax_merged['PARID'].isin(unmerged_not_present)]
unmerged_present_df = sales_tax_merged[sales_tax_merged['PARID'].isin(unmerged_present)]

print(unmerged_not_present_df.shape)
print(unmerged_present_df.shape)

(13, 114)
(2116, 114)


In [125]:
unmerged_present_df["year_before"] = unmerged_present_df["SALE_YR"] - 1
unmerged_present_df["year_after"] = unmerged_present_df["SALE_YR"] + 1

unmerged_year_status_df = pd.merge(unmerged_present_df, tax_digest_df[["PARID", "TAXYR"]], left_on=["PARID", "year_before"], right_on=["PARID", "TAXYR"], how="left", suffixes=("", "_before"))
unmerged_year_status_df = pd.merge(unmerged_year_status_df, tax_digest_df[["PARID", "TAXYR"]], left_on=["PARID", "year_after"], right_on=["PARID", "TAXYR"], how="left", suffixes=("", "_after"))

/var/folders/bb/g7vlcgfn14ld8_w41331g0dw0000gn/T/ipykernel_15328/1611325224.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unmerged_present_df["year_before"] = unmerged_present_df["SALE_YR"] - 1
/var/folders/bb/g7vlcgfn14ld8_w41331g0dw0000gn/T/ipykernel_15328/1611325224.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unmerged_present_df["year_after"] = unmerged_present_df["SALE_YR"] + 1


In [131]:
before = (~unmerged_year_status_df.TAXYR_before.isna()).sum()
after = (~unmerged_year_status_df.TAXYR_after.isna()).sum()
both = ((~unmerged_year_status_df.TAXYR_before.isna()) & (~unmerged_year_status_df.TAXYR_after.isna())).sum()
neither = ((unmerged_year_status_df.TAXYR_before.isna()) & (unmerged_year_status_df.TAXYR_after.isna())).sum()

print(f"{before}/{unmerged_present_df.shape[0]} parcels existed before sale\n{after}/{unmerged_present_df.shape[0]} parcels existed after \n{both}/{unmerged_present_df.shape[0]} parcels existed both before and after sale.\n" + 
      f"{neither}/{unmerged_present_df.shape[0]} parcels existed neither before nor after.")

645/2116 parcels existed before sale
2028/2116 parcels existed after 
628/2116 parcels existed both before and after sale.
71/2116 parcels existed neither before nor after.


In [132]:
neither_df = unmerged_year_status_df[((unmerged_year_status_df.TAXYR_before.isna()) & (unmerged_year_status_df.TAXYR_after.isna()))]

In [133]:
neither_df

,PARID,BOOK,PAGE,OLDOWN,OWN1_x,SALEDT,PRICE,SALETYPE,INSTRTYP,SALEVALDESCR,...,owner_mail_paddr1,owner_mail_paddr2,owner_mail_paddr3,owner_street_part,property_street_part,rental_flag,year_before,year_after,TAXYR_before,TAXYR_after
20,18 299 07 010,23550,00552,CATALINA VENTURES LLC,HARRY NULLET/BNP PARIBAS,2013-01-31,1300000,NaN,LW,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2012,2014,NaN,NaN
21,18 299 07 009,23550,NaN,CATALINA VENTURES LLC,HARRY NULLET/BNP PARIBAS,2013-01-31,1300000,NaN,LW,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2012,2014,NaN,NaN
22,18 299 07 009,23550,00552,CATALINA VENTURES LLC,HARRY NULLET/BNP PARIBAS,2013-01-31,1300000,NaN,LW,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2012,2014,NaN,NaN
23,18 298 08 021,23550,NaN,CATALINA VENTURES LLC,CHAMBLEE KDP REALTY LLC,2013-01-31,1300000,NaN,LW,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2012,2014,NaN,NaN
24,18 298 08 017,23550,NaN,CATALINA VENTURES LLC,CHAMBLEE KDP REALTY LLC,2013-01-31,1300000,NaN,LW,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2012,2014,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421,15 062 07 056,24170,00106,5065 FLAT SHOALS PARKWAY TRUST,GRIFFIN ANTHONY J SR,2013-12-11,0,NaN,WD,0,...,3632 RIVER CLIFF CT,NaN,DECATUR GA 30034,3632 RIVER CLIFF CT,5065 FLAT SHOALS PKWY,1.0,2012,2014,NaN,NaN
699,16 010 01 196,24421,NaN,REO FUNDING SOLUTIONS II LLC,D R HORTON CROWN LLC,2014-06-06,154000,NaN,LW,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2013,2015,NaN,NaN
1423,18 301 02 111,26441,00597,HOUSING GROUP INC,MASHBURN KIMBERLY,2017-08-17,635000,,LW,0,...,NaN,NaN,NaN,NaN,NaN,NaN,2016,2018,NaN,NaN
1424,18 301 02 111,26441,00596,HOUSING DEVELOPMENT CORPORATION OF,HOUSING GROUP INC,2017-08-17,295871,,LW,0,...,NaN,NaN,NaN,NaN,NaN,NaN,2016,2018,NaN,NaN
